# FIREGUARD ID v2 — Minggu 2: [L] Data Preparation, Spatial Grid & Feature Governance
**Sesuai Ketentuan Minggu 2:** Exact polygon clipping, grid 0.1°, feature engineering, leakage-free scaling, spatial-block splits, dan ringkasan Apa–Mengapa–Bagaimana.

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import json
import numpy as np
import pandas as pd
from src.utils import load_yaml, resolve_path

proj_cfg = load_yaml("config/project.yaml", repo_root=REPO_ROOT)
print("Repo root:", REPO_ROOT)

Repo root: FIREGUARD-ID


## 1. Skema Fitur Terkunci (`features_baseline.yaml` vs `features_extended.yaml`) & Leakage Guard

In [2]:
from src.data.quality import validate_no_leakage_columns
base_cfg = load_yaml("config/features_baseline.yaml", repo_root=REPO_ROOT)
ext_cfg = load_yaml("config/features_extended.yaml", repo_root=REPO_ROOT)
validate_no_leakage_columns(base_cfg["clustering_features"], base_cfg["forbidden_features"])
print("Baseline locked features:", base_cfg["clustering_features"])
print("Extended candidate features:", ext_cfg["clustering_features"])

Baseline locked features: ['hotspot_density_km2', 'fire_days', 'active_years_count', 'frp_total_density_mw_km2']
Extended candidate features: ['hotspot_density_km2', 'fire_days', 'active_years_count', 'frp_total_density_mw_km2', 'elevation_m', 'era5_dry_t2m_mean_c', 'era5_dry_soil_moisture_mean', 'era5_annual_precip_mm']


## 2. Verifikasi Grid 853 Sel, Zero-Detection NaNs & Spatial-Block Folds

In [3]:
df_ext = pd.read_parquet(resolve_path("data/processed/riau_grid_extended.parquet", REPO_ROOT))
print("Shape:", df_ext.shape, "| Total hotspots:", int(df_ext["hotspot_count"].sum()))
print("Zero-detection cells (conditional FRP kept NaN):", int(df_ext["has_zero_detections"].sum()))
print("Spatial blocks:", df_ext["spatial_block"].value_counts().to_dict())
display(df_ext[base_cfg["clustering_features"]].describe().T.round(4))

Shape: (853, 44) | Total hotspots: 15510
Zero-detection cells (conditional FRP kept NaN): 249
Spatial blocks: {'SE': 293, 'NW': 281, 'NE': 150, 'SW': 129}


,count,mean,std,min,25%,50%,75%,max
hotspot_density_km2,853.0,0.1789,0.4650,0.0,0.0,0.0325,0.1410,7.0886
fire_days,853.0,8.5287,25.9752,0.0,0.0,3.0000,9.0000,472.0000
active_years_count,853.0,2.0164,1.7702,0.0,0.0,2.0000,4.0000,5.0000
frp_total_density_mw_km2,853.0,1.0840,3.1761,0.0,0.0,0.1183,0.5829,42.1954
